# Sentiment Analysis using RNN

We will build a Recurrent Neural Network (RNN) to predict the sentiment of car reviews from our choice of data. 

Suggested pipeline: data loading, preprocessing, model building, training, and evaluation.

Workshop Agenda: Classify reviews as positive or negative.

**Steps**:
1. Load and Explore Data
2. Preprocessing (Tokenization, Vocabulary, Padding)
3. Create PyTorch Datasets and DataLoaders
4. Define RNN Model Architecture
5. Train the Model
6. Evaluate Performance
7. Inference on New Data

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, Dataset, DataLoader

import pandas as pd
import numpy as np
from collections import Counter
import re
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Set random seed for reproducibility
SEED = 1234
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Data

In [ ]:
# Load the dataset
data_path = 'data/car_review.csv'
df = pd.read_csv(data_path)

# Display first few rows
print(df.head())
print('-' * 50)

print(df.info()) # Check the data. Do Necessary Preprocessing.

In [ ]:
# Columns from the dataset
TEXT_COL = 'review_text'  
LABEL_COL = 'label' 

# Check class distribution
print(df[LABEL_COL].value_counts())

In [ ]:
df.describe()

In [ ]:
df.info()

## 3. Preprocessing

Neural networks cannot understand raw text. We need to convert text into numbers. 

We will follow these steps:
1. **Tokenization**: Split text into words/tokens.
2. **Vocabulary Building**: Create a mapping from word to unique integer index.
3. **Encoding**: Convert reviews to lists of integers.
4. **Padding**: Ensure all sequences have the same length.

In [ ]:
def tokenize(text):
    """Simple whitespace tokenizer that removes non-alphanumeric characters."""
    text = text.lower()
    # Remove special chars
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.split()

In [ ]:
# Build Vocabulary

all_words = []
for review in df[TEXT_COL]:
    all_words.extend(tokenize(str(review)))

word_counts = Counter(all_words)
# Keep only most common words to keep model simple
MAX_VOCAB_SIZE = 5000
most_common_words = word_counts.most_common(MAX_VOCAB_SIZE)

In [ ]:
print(len(all_words), Counter(all_words))

### Create mapping: Word - to -  Index (Why not use pre-trained embeddings?)

In [ ]:
vocab = {word: i + 2 for i, (word, _) in enumerate(most_common_words)}

vocab['<PAD>'] = 0 # Padding for reviews of different lengths (Why? padding is used to make all reviews of same length)
vocab['<UNK>'] = 1 # Unknown words (Why? because we are not using any pre-trained embeddings)

print(f"Vocabulary size: {len(vocab)}")
print(f"Most common words: {most_common_words[:5]}")
print(f"Vocabulary: {vocab['the']}")
print(f"Vocabulary: {list(vocab.keys())}")

In [ ]:
# Encode function
def encode_text(text, vocab, max_length=100):
    tokens = tokenize(str(text))
    # Convert tokens to indices
    indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    
    # Padding / Truncating
    if len(indices) < max_length:
        # Pad with 0s
        indices = indices + [vocab['<PAD>']] * (max_length - len(indices))
    else:
        # Truncate
        indices = indices[:max_length]
        
    return indices

### Convert labels to integers if they are strings (e.g., 'Pos' as 1, 'Neg' as 0)
- This depends on dataset values. We assume 1/0 or Pos/Neg

In [ ]:
# Example mapping (Customize this!)
label_map = {label: i for i, label in enumerate(df[LABEL_COL].unique())}
print(f"Label Mapping: {label_map}")

# Apply encoding
MAX_SEQ_LEN = 100

X = [encode_text(text, vocab, MAX_SEQ_LEN) for text in df[TEXT_COL]]
y = [label_map[label] for label in df[LABEL_COL]]

X = np.array(X)
y = np.array(y)

## 4. PyTorch Dataset & DataLoaders (Optional? If yes, then how?)

In [ ]:
# class CarReviewDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = torch.tensor(X, dtype=torch.long)
#         self.y = torch.tensor(y, dtype=torch.long) # Use float for BCEWithLogitsLoss if binary, long for CrossEntropy
        
#     def __len__(self):
#         return len(self.X)
    
#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# load data into torch dataset using TensorDataset
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.long))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## 5. RNN Model Architecture

Define a simple RNN class.
- **Embedding Layer**: Converts integer indices into dense vectors.
- **RNN Layer**: Processes the sequence of embeddings.
- **Linear Layer**: Maps the final hidden state to the output class (Sentiment).

In [ ]:
class SentimentRNN(nn.Module):
    """
    A simple RNN model for sentiment analysis
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        """
        Initialize the model: vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout
        """
        super(SentimentRNN, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Swap this for LSTM or GRU
        # batch_first=True here input shape is (batch, seq_len, features)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, text):
        # text shape: [batch size, seq len]
        
        embedded = self.embedding(text)
        # embedded shape: [batch size, seq len, emb dim]
        
        output, hidden = self.rnn(embedded)
        # output shape: [batch size, seq len, hidden dim]
        # hidden shape: [n layers, batch size, hidden dim]
        
        # We use the final hidden state from the last layer
        return self.fc(hidden[-1])

## 6. Training Configuration

In [ ]:
# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = len(label_map) # Number of classes
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 0.001
EPOCHS = 10

# Initialize Model
model = SentimentRNN(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT)
model = model.to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 7. Training Loop

In [ ]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for texts, labels in iterator:
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(texts)
        loss = criterion(predictions, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Accuracy
        _ , predicted = torch.max(predictions.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    return epoch_loss / len(iterator), correct / total

In [ ]:
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels in iterator:
            texts, labels = texts.to(device), labels.to(device)
            
            predictions = model(texts)
            loss = criterion(predictions, labels)
            
            epoch_loss += loss.item()
            
            _, predicted = torch.max(predictions.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    return epoch_loss / len(iterator), correct / total

In [ ]:
# Run Training
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

# TASK: Plot the training and test loss and accuracy

## 8. Inference

In [ ]:
def predict_sentiment(text):
    model.eval()
    encoded = encode_text(text, vocab, MAX_SEQ_LEN)
    tensor = torch.tensor(encoded).unsqueeze(0).to(device)
    
    with torch.no_grad():
        prediction = model(tensor)
        _, predicted_class = torch.max(prediction, 1)
        
    # Inverse map to get label string
    inv_map = {v: k for k, v in label_map.items()}
    return inv_map[predicted_class.item()]

In [ ]:
# Test on a few examples
test_reviews = [
    "This car is amazing and very fast.",
    "This car is worst and very fast but not so good.",
    "The engine makes a terrible noise.",
    "I love the interior design.",
    "Waste of money, very poor mileage.",
]

for review in test_reviews:
    print(f"Review: {review} | Prediction: {predict_sentiment(review)}")

In [ ]:
test_sentences = [
    # TEST CASES
    "I just wanted to warn my fellow customers who still deciding which way to go about that fact.",
    "Car is good but i dont like another color"
]

for sentence in test_sentences:
    print(f"Sentence: {sentence[:100]} | Prediction: {predict_sentiment(sentence)}")

### TODO
- Experiment with different word embeddings: Try using pre-trained embeddings (e.g., GloVe, Word2Vec) instead of training them from scratch, or compare the performance of different embedding dimensions.
- Compare RNN architectures: Implement and compare the performance of SimpleRNN, GRU, and LSTM layers. Discuss the advantages and disadvantages of each.
- Hyperparameter tuning: Experiment with different learning rates, batch sizes, number of epochs, and dropout rates to optimize model performance.
- Analyze the impact of text preprocessing: Explore how different preprocessing steps (e.g., stemming, lemmatization, stop-word removal, different tokenizers) affect the models accuracy and training time.

- **Introduce an attention mechanism**: Add a simple attention layer to the RNN model and observe its effect on performance and potentially interpretability.


# Conclusion

We have built and trained a Recurrent Neural Network (RNN) for sentiment analysis. 

To further improve the model accuracy, students can explore hyperparameter tuning (e.g., learning rate, batch size, number of epochs, hidden layer size, dropout rates), 